# Presentation figure: HighResMIP MME biases vs. CM2.5-FLOR bias and topography sensitivity

1x5 JAS precipitation panels on one uniform domain, sharing one colorbar, with the NAM sub-domain polygons on every panel:

| A | B | C | D | E |
|---|---|---|---|---|
| HighResMIP Low-Res. MME − IMERG | HighResMIP High-Res. MME − IMERG | FLOR CTRL − IMERG | FLOR HI$_{gbl}$ − CTRL | FLOR HI$_{mex}$ − CTRL |

**Prerequisite:** run `HighResMIP_prec_historical_biases.ipynb` once so the per-config bias cache
(`../highresmip_diagnostics/cache/pr_bias_cons2_{cfg}.nc`) exists. This notebook reads that cache rather than recomputing.

In [ ]:
import os, sys
import warnings
warnings.filterwarnings('ignore')

import xesmf as xe
import xarray as xr
xr.set_options(keep_attrs=True)
import numpy as np
np.seterr(divide='ignore', invalid='ignore')

import cartopy
cartopy.config['data_dir'] = "/discover/nobackup/projects/jh_tutorials/JH_examples/JH_datafiles/Cartopy"
cartopy.config['pre_existing_data_dir'] = "/discover/nobackup/projects/jh_tutorials/JH_examples/JH_datafiles/Cartopy"
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

import matplotlib as mpl
import matplotlib.patches as patches
import matplotlib.pyplot as plt

%config InlineBackend.figure_format = 'retina'

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path + "/py_functions")
from colorbar_funcs import *
from data_funcs import *
from stats_funcs import *
from domain_funcs import *

dpath0 = '/discover/nobackup/projects/giss/baldwin_nip/dmkumar'  # top level data directory

## 1. HighResMIP low-/high-res. MME bias vs. IMERG (from cache)

In [ ]:
# --- HRMIP configuration maps (same as HighResMIP_prec_historical_biases) --- #
models = ['CAM-MPAS','CAMS-CSM1-0','CMCC-CM2','CNRM-CM6-1','EC-Earth3P','ECMWF-IFS','FGOALS-f3','GFDL-CM4',
          'HadGEM3-GC31','HiRAM-SIT','INM-CM5','IPSL-CM6A','MPI-ESM1-2','MRI-ESM2','NICAM16']

lr_map = {
    'CAM-MPAS': ['CAM-MPAS-LR'], 'CAMS-CSM1-0': ['CAMS-CSM1-0-LR'], 'CMCC-CM2': ['CMCC-CM2-HR4'],
    'CNRM-CM6-1': ['CNRM-CM6-1'], 'EC-Earth3P': [], 'ECMWF-IFS': ['ECMWF-IFS-LR'], 'FGOALS-f3': ['FGOALS-f3-L'],
    'GFDL-CM4': ['GFDL-CM4'], 'HadGEM3-GC31': ['HadGEM3-GC31-LM'], 'HiRAM-SIT': [], 'INM-CM5': ['INM-CM5-0'],
    'IPSL-CM6A': ['IPSL-CM6A-LR'], 'MPI-ESM1-2': ['MPI-ESM1-2-LR'], 'MRI-ESM2': ['MRI-ESM2-0'], 'NICAM16': [],
}
hr_map = {
    'CAM-MPAS': ['CAM-MPAS-HR'], 'CAMS-CSM1-0': ['CAMS-CSM1-0'], 'CMCC-CM2': ['CMCC-CM2-VHR4'],
    'CNRM-CM6-1': ['CNRM-CM6-1-HR'], 'EC-Earth3P': ['EC-Earth3P', 'EC-Earth3P-HR'], 'ECMWF-IFS': ['ECMWF-IFS-HR'],
    'FGOALS-f3': ['FGOALS-f3-H'], 'GFDL-CM4': ['GFDL-CM4C192'], 'HadGEM3-GC31': ['HadGEM3-GC31-MM', 'HadGEM3-GC31-HM'],
    'HiRAM-SIT': ['HiRAM-SIT-LR', 'HiRAM-SIT-HR'], 'INM-CM5': ['INM-CM5-H'], 'IPSL-CM6A': ['IPSL-CM6A-ATM-HR'],
    'MPI-ESM1-2': ['MPI-ESM1-2-HR', 'MPI-ESM1-2-XR'], 'MRI-ESM2': ['MRI-AGCM3-2-H', 'MRI-AGCM3-2-S'],
    'NICAM16': ['NICAM16-7S', 'NICAM16-8S'],
}

prod = 'imerg'
cache_dir = '../highresmip_diagnostics/cache/'

# --- load per-config IMERG bias maps from the cache --- #
hr_bias = {'lr': {}, 'hr': {}}
nlon = {}   # full-grid longitude count per config (stored as a cache attribute)
for res, rmap in {'lr': lr_map, 'hr': hr_map}.items():
    for model in models:
        for cfg in rmap[model]:
            f = os.path.join(cache_dir, f'pr_bias_cons2_{cfg}.nc')
            if not os.path.exists(f):
                raise FileNotFoundError(f'{f} missing - run HighResMIP_prec_historical_biases.ipynb first.')
            _ds = xr.open_dataset(f)
            hr_bias[res][cfg] = _ds[prod].load()
            nlon[cfg] = int(_ds.attrs['nlon'])

# --- MME: conservatively regrid every member to the lowest-res. model grid of its set, then average --- #
res_key = {'lr': 'IPSL-CM6A-LR', 'hr': 'IPSL-CM6A-ATM-HR'}
mme, mme_agree = {}, {}
agree_thresh = 0.6   # stipple where fewer than this fraction of members share the MME-mean sign
for res, rmap in {'lr': lr_map, 'hr': hr_map}.items():
    target = hr_bias[res][res_key[res]]
    stack = xr.concat([conservative_regrid(hr_bias[res][cfg].assign_coords(source_id=cfg), target.lat, target.lon)
                       for model in models for cfg in rmap[model]], dim='source_id')
    mme[res] = stack.mean(dim='source_id')
    mme_agree[res] = sign_agreement(stack, dim='source_id')
    print(f'{res.upper()} MME: {stack.sizes["source_id"]} configs on a {target.sizes["lat"]}x{target.sizes["lon"]} grid')

## 1b. ETOPO5 at each ensemble's median resolution

Following `topo_resolution_schematic.ipynb`: ETOPO5 is conservatively regridded (xESMF, `skipna=True`, `na_thres=0.75`)
onto a regular global grid whose spacing is 360 / (median `nlon` of the ensemble), one grid per resolution set.

In [ ]:
median_nlon = {res: float(np.median([nlon[cfg] for cfg in hr_bias[res]])) for res in ['lr', 'hr']}
median_deg  = {res: 360.0 / median_nlon[res] for res in median_nlon}
print({res: f'nlon={median_nlon[res]:g} -> {median_deg[res]:.2f} deg' for res in median_nlon})

etopo_full = xr.open_dataset(f'{dpath0}/topo_files/obs.etopo5.zsurf.nc').ROSE.rename({'ETOPO05_Y': 'lat', 'ETOPO05_X': 'lon'})
lat_fine, lon_fine = etopo_full.lat, etopo_full.lon
dlat_fine = float(lat_fine[1] - lat_fine[0]); dlon_fine = float(lon_fine[1] - lon_fine[0])
etopo_src = etopo_full.to_dataset(name='topo')
etopo_src['lat_b'] = xr.DataArray(np.append(lat_fine.values - dlat_fine/2, lat_fine.values[-1] + dlat_fine/2), dims=['lat_b'])
etopo_src['lon_b'] = xr.DataArray(np.append(lon_fine.values - dlon_fine/2, lon_fine.values[-1] + dlon_fine/2), dims=['lon_b'])

topo_med = {}
for res, deg in median_deg.items():
    lat_c = np.arange(lat_fine.min(), lat_fine.max() + 0.01, deg)
    lon_c = np.arange(lon_fine.min(), lon_fine.max() + 0.01, deg)
    grid = xr.Dataset({'lat': (['lat'], lat_c), 'lon': (['lon'], lon_c),
                       'lat_b': (['lat_b'], np.append(lat_c - deg/2, lat_c[-1] + deg/2)),
                       'lon_b': (['lon_b'], np.append(lon_c - deg/2, lon_c[-1] + deg/2))})
    regridder = xe.Regridder(etopo_src, grid, method='conservative', periodic=True)
    topo_med[res] = regridder(etopo_src['topo'], skipna=True, na_thres=0.75).sel(lat=slice(10, 50), lon=slice(230, 270))

## 2. CM2.5-FLOR: CTRL bias vs. IMERG and topography sensitivity

In [ ]:
flor_runs = ['ctrl', 'hitopo', 'hicam']       # hitopo = HI_gbl, hicam = HI_mex
latmin, latmax = 10, 70
lonmin, lonmax = 200, 300
yr_min, yr_max = '0101', '0200'

# --- IMERG --- #
ds = xr.open_dataset(f'{dpath0}/obs_data/prec/imerg.gn.timeseries.2001-2018.nc', chunks={}).precipitation * 24  # mm/hr -> mm/day
ds = lonFlip(ds.transpose('time', 'lat', 'lon'))
imerg = ds.sel(lat=slice(latmin, latmax), lon=slice(lonmin, lonmax))

# --- FLOR precip --- #
flor = {}
for run in flor_runs:
    ds = xr.open_dataset(f'{dpath0}/FLOR/{run}/pi/flor.{run}.precip.monthly.nc').precip.sel(time=slice(yr_min, yr_max)) * 86400
    ds = ds.rename({'grid_xt': 'lon', 'grid_yt': 'lat'})
    flor[run] = ds.sel(lon=slice(lonmin, lonmax), lat=slice(latmin, latmax))

# --- FLOR model topography (the HRMIP panels use ETOPO5 at the ensemble median resolution, see 1b) --- #
topo = {}
for run in flor_runs:
    ds = xr.open_dataset(f'{dpath0}/topo_files/flor.{run}.zsurf.nc').ZSURF.rename({'GRID_XT': 'lon', 'GRID_YT': 'lat'})
    topo[run] = ds.where(ds > 0, np.nan).sel(lat=slice(latmin, latmax), lon=slice(lonmin, lonmax))

# --- JAS means (seasonal and yearly), IMERG regridded conservatively to the FLOR grid --- #
flor_jas, flor_ann = {}, {}
for run in flor_runs:
    flor_jas[run] = jas_seasonal_mean(flor[run]).load()
    flor_ann[run] = jas_yearly_mean(flor[run]).load()
lats, lons = flor['ctrl'].lat, flor['ctrl'].lon
imerg_jas = conservative_regrid(jas_seasonal_mean(imerg).load(), lats, lons)
imerg_ann = conservative_regrid(jas_yearly_mean(imerg).load(), lats, lons)

# --- CTRL - IMERG (masked where not significant) --- #
ctrl_bias, ctrl_bias_mask, _ = sigtest2n(flor_ann['ctrl'], imerg_ann, flor_jas['ctrl'], imerg_jas)

# --- HI - CTRL, and where the HI run reduces the |bias| significantly (hatched) --- #
topo_diff, topo_diff_mask, bias_reduced = {}, {}, {}
run_bias = {'ctrl': ctrl_bias}
for run in ['hitopo', 'hicam']:
    run_bias[run], _, _ = sigtest2n(flor_ann[run], imerg_ann, flor_jas[run], imerg_jas)
    d, m, _ = sigtest2n(flor_ann[run], flor_ann['ctrl'], flor_jas[run], flor_jas['ctrl'])
    topo_diff[run], topo_diff_mask[run] = d, m
    change = np.abs(run_bias[run]) - np.abs(ctrl_bias)
    bias_reduced[run] = change.where(m.notnull())
print('Done.')

## 3. Figure

In [ ]:
# ---------------- settings ---------------- #
trans = ccrs.PlateCarree()
proj  = ccrs.PlateCarree()
map_bnds = [240, 260, 15, 40]          # uniform domain for all five panels (lon 240-260E = 120-100W)

dcmap, _, _, _ = get_settings(field='precip', diff=True)
dlevels = np.linspace(-3, 3, 25)
dnorm = mpl.colors.BoundaryNorm(dlevels, dcmap.N)

zlevels = np.linspace(800, 3800, 7)    # topography contours [m]
coords_map, regions = nam_regions()

titles = ['Low-Res. MME$-$IMERG', 'High-Res. MME$-$IMERG', 'CTRL$-$IMERG',
          r'HI$_{\mathbf{gbl}}$$-$CTRL', r'HI$_{\mathbf{mex}}$$-$CTRL']
letters = list('ABCDE')
title_kw  = {'weight': 'bold', 'size': 20}
letter_kw = {'weight': 'bold', 'size': 26, 'ha': 'left', 'va': 'bottom'}
tick_sz = 16
poly_lw = 3.5

# (field, lon, lat, topography, hatch field) for each panel
panels = [
    (mme['lr'],                 mme['lr'].lon, mme['lr'].lat, topo_med['lr'], None),
    (mme['hr'],                 mme['hr'].lon, mme['hr'].lat, topo_med['hr'], None),
    (ctrl_bias_mask,            lons,          lats,          topo['ctrl'],  None),
    (topo_diff_mask['hitopo'],  lons,          lats,          topo['hitopo'], bias_reduced['hitopo']),
    (topo_diff_mask['hicam'],   lons,          lats,          topo['hicam'],  bias_reduced['hicam']),
]

agree = {0: mme_agree['lr'], 1: mme_agree['hr']}   # panel index -> MME sign agreement

# ---------------- plot ---------------- #
fig, axes = plt.subplots(nrows=1, ncols=5, figsize=(26, 8.2), layout='constrained', subplot_kw={'projection': proj})

for i, (ax, (field, lo, la, tp, hatch)) in enumerate(zip(axes, panels)):
    cf = ax.pcolormesh(lo, la, field, cmap=dcmap, norm=dnorm, transform=trans)
    ax.contour(tp.lon, tp.lat, tp, levels=zlevels, linewidths=1.4, colors='k', transform=trans)
    if i in agree:          # stipple where < agree_thresh of the models agree on the sign of the MME bias
        low = (agree[i] < agree_thresh)
        if bool(low.any()):
            ax.contourf(low.lon, low.lat, low.astype(float), levels=[0.5, 1.5], colors='none', hatches=['....'],
                        transform=trans, zorder=104)
    if hatch is not None:   # significant reduction of the IMERG bias relative to CTRL
        ax.contourf(hatch.lon, hatch.lat, hatch, 0, colors='none', hatches=['', '///'], extend='lower',
                    zorder=100, transform=trans)

    # NAM sub-domain polygons (identical on every panel)
    ax.add_patch(patches.Polygon(coords_map['north'], closed=True, ec='firebrick', fc='none', lw=poly_lw, ls='--',
                                 transform=trans, zorder=105))
    ax.add_patch(patches.Polygon(coords_map['south'], closed=True, ec='red', fc='none', lw=poly_lw, ls='--',
                                 transform=trans, zorder=105))

    ax.coastlines(color='k', linewidth=1.2)
    ax.set_extent(map_bnds, crs=trans)
    ax.set_title(titles[i], pad=10, **title_kw)
    ax.text(0.01, 1.075, letters[i], transform=ax.transAxes, **letter_kw)

    gl = ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.xlocator = mpl.ticker.FixedLocator(range(-120, -99, 5))
    gl.ylocator = mpl.ticker.FixedLocator(range(15, 41, 5))
    gl.bottom_labels = True; gl.top_labels = False; gl.right_labels = False
    gl.left_labels = (i == 0)
    gl.xformatter = LONGITUDE_FORMATTER; gl.yformatter = LATITUDE_FORMATTER
    gl.xlabel_style = {'size': tick_sz}; gl.ylabel_style = {'size': tick_sz}

# ---------------- single shared colorbar ---------------- #
cbar = fig.colorbar(cf, ax=axes, location='right', ticks=[-3, -2, -1, 0, 1, 2, 3], extend='both',
                    shrink=0.8, pad=0.015, aspect=30)
cbar.set_label('$\\Delta$ Precipitation [mm/day]', size=22, labelpad=15)
cbar.ax.tick_params(labelsize=20)

fig.text(0.005, 0.005, 'JAS mean. A,B: stippling = <60% of models agree on bias sign; contours = ETOPO5 at the ensemble median resolution. '
         'C-E: masked where p>0.1; hatching = significant reduction of the CTRL bias vs. IMERG; contours = FLOR topography. '
         'Contours every 500 m from 800 m.', size=12, ha='left', va='bottom', alpha=0.7)

# ---------------- output ---------------- #
plt.savefig('../figs/prec_presentation_hrmip_mme_flor_5panel.pdf', transparent=False, bbox_inches='tight')
plt.savefig('../figs/prec_presentation_hrmip_mme_flor_5panel.png', transparent=False, bbox_inches='tight', dpi=300)